# Student depression prediction

## Data Analysis

In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv('Data/Student_Depression_Dataset.csv')

In [5]:
df.head(5)

,id,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,2,Male,33.0,Visakhapatnam,Student,5.0,0.0,8.97,2.0,0.0,5-6 hours,Healthy,B.Pharm,Yes,3.0,1.0,No,1
1,8,Female,24.0,Bangalore,Student,2.0,0.0,5.90,5.0,0.0,5-6 hours,Moderate,BSc,No,3.0,2.0,Yes,0
2,26,Male,31.0,Srinagar,Student,3.0,0.0,7.03,5.0,0.0,Less than 5 hours,Healthy,BA,No,9.0,1.0,Yes,0
3,30,Female,28.0,Varanasi,Student,3.0,0.0,5.59,2.0,0.0,7-8 hours,Moderate,BCA,Yes,4.0,5.0,Yes,1
4,32,Female,25.0,Jaipur,Student,4.0,0.0,8.13,3.0,0.0,5-6 hours,Moderate,M.Tech,Yes,1.0,1.0,No,0


In [6]:
df['Sleep Duration'].value_counts()

Sleep Duration
Less than 5 hours    8310
7-8 hours            7346
5-6 hours            6183
More than 8 hours    6044
Others                 18
Name: count, dtype: int64

In [7]:
df["Work Pressure"].value_counts()

Work Pressure
0.0    27898
5.0        2
2.0        1
Name: count, dtype: int64

In [8]:
drop_columns = ['id', 'Work Pressure', 'Profession', 'Job Satisfaction' ]
df.drop(columns=drop_columns, inplace=True)

In [9]:
df.isnull().sum()

Gender                                   0
Age                                      0
City                                     0
Academic Pressure                        0
CGPA                                     0
Study Satisfaction                       0
Sleep Duration                           0
Dietary Habits                           0
Degree                                   0
Have you ever had suicidal thoughts ?    0
Work/Study Hours                         0
Financial Stress                         3
Family History of Mental Illness         0
Depression                               0
dtype: int64

In [33]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 27901 entries, 0 to 27900
Data columns (total 14 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Gender                                 27901 non-null  str    
 1   Age                                    27901 non-null  float64
 2   City                                   27901 non-null  str    
 3   Academic Pressure                      27901 non-null  float64
 4   CGPA                                   27901 non-null  float64
 5   Study Satisfaction                     27901 non-null  float64
 6   Sleep Duration                         27901 non-null  str    
 7   Dietary Habits                         27901 non-null  str    
 8   Degree                                 27901 non-null  str    
 9   Have you ever had suicidal thoughts ?  27901 non-null  str    
 10  Work/Study Hours                       27901 non-null  float64
 11  Financial Str

In [10]:
X = df.drop(labels="Depression", axis=1)
y = df['Depression']

In [11]:
num_columns = ['Age', 'Academic Pressure', 'CGPA', 'Study Satisfaction','Work/Study Hours', 'Financial Stress']

## Data Preprocessing

In [12]:
from sklearn.impute import SimpleImputer ## HAndling Missing Values
from sklearn.preprocessing import StandardScaler # HAndling Feature Scaling
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder # Ordinal Encoding
## pipelines
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [13]:
num_pipeline = Pipeline(
    steps=[
        ('Imputer', SimpleImputer()),
        ('StandardScaler', StandardScaler())
    ]
)

In [14]:
cat_pipeline = Pipeline(
    steps=[
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]
)

In [15]:
sleep_duration =['Less than 5 hours', '5-6 hours', '7-8 hours', 'More than 8 hours', 'Others']
health_situation = ['Unhealthy', 'Moderate', 'Healthy', 'Others']
cat_features = [
    'Gender',
    'City',
    'Degree',
    'Have you ever had suicidal thoughts ?',
    'Family History of Mental Illness'
]

ordinal_cat = ['Sleep Duration','Dietary Habits']

In [16]:
ordinal_pipeline=Pipeline(
    
    steps=[
        ('ordinalencoder',OrdinalEncoder(categories=[sleep_duration, health_situation])),
    ]
    
)

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num_pipeline', num_pipeline, num_columns),
        ('ordinal_pipeline', ordinal_pipeline, ordinal_cat),
        ('cat_pipeline', cat_pipeline, cat_features)
    ]
)

In [18]:


from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.30,random_state=30)


In [19]:
X_train_transformed = preprocessor.fit_transform(X_train)

print(type(X_train_transformed))
print(X_train_transformed.shape)
print(len(preprocessor.get_feature_names_out()))

<class 'scipy.sparse._csr.csr_matrix'>
(19530, 90)
90


In [20]:
X_train=pd.DataFrame(preprocessor.fit_transform(X_train).toarray(),columns=preprocessor.get_feature_names_out())
X_test=pd.DataFrame(preprocessor.transform(X_test).toarray(),columns=preprocessor.get_feature_names_out())

## Model Training

In [21]:
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn import linear_model
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import GradientBoostingRegressor

In [22]:
modelmlg = LinearRegression()
modeldcr = DecisionTreeRegressor()
modelbag = BaggingRegressor()
modelrfr = RandomForestRegressor()
modelSVR = SVR()
modelKNN = KNeighborsRegressor(n_neighbors=5)
modelETR = ExtraTreesRegressor()
modelRE=Ridge()
modelLO=linear_model.Lasso(alpha=0.1)

modelGBR = GradientBoostingRegressor(loss='squared_error', learning_rate=0.1, n_estimators=100, subsample=1.0,
                                     criterion='friedman_mse', min_samples_split=2, min_samples_leaf=1,
                                     min_weight_fraction_leaf=0.0, max_depth=3, min_impurity_decrease=0.0,
                                     init=None, random_state=None, max_features=None,
                                     alpha=0.9, verbose=0, max_leaf_nodes=None, warm_start=False,
                                     validation_fraction=0.1, n_iter_no_change=None, tol=0.0001, ccp_alpha=0.0)



In [23]:
import pandas as pd

Results = pd.DataFrame(columns=[
    'Model',
    'Mean_Absolute_Error_MAE',
    'Mean_Squared_Error_MSE',
    'Root_Mean_Squared_Error_RMSE',
    'R2_Score',
    'Root_Mean_Squared_Log_Error_RMSLE',
    'Mean_Absolute_Percentage_Error_MAPE',
    'Adjusted_R2_Score'
])

In [24]:


def MAPE (y_test, y_pred):
    y_test, y_pred = np.array(y_test), np.array(y_pred)
    return np.mean(np.abs((y_test - y_pred) / y_test)) * 100

MM = [modelmlg, modeldcr, modelrfr, modelKNN, modelETR, modelGBR, modelbag,modelRE,modelLO]

for models in MM:
    
    
    models.fit(X_train, y_train)
    

    y_pred = models.predict(X_test)
    
    
    print('Model Name: ', models)
    

    from sklearn import metrics

    print('Mean Absolute Error (MAE):', round(metrics.mean_absolute_error(y_test, y_pred),3))  
    print('Mean Squared Error (MSE):', round(metrics.mean_squared_error(y_test, y_pred),3))  
    print('Root Mean Squared Error (RMSE):', round(np.sqrt(metrics.mean_squared_error(y_test, y_pred)),3))
    print('R2_score:', round(metrics.r2_score(y_test, y_pred),6))
    print('Root Mean Squared Log Error (RMSLE):', round(np.log(np.sqrt(metrics.mean_squared_error(y_test, y_pred))),3))
    
    result = MAPE(y_test, y_pred)
    print('Mean Absolute Percentage Error (MAPE):', round(result, 2), '%')
    

    r_squared = round(metrics.r2_score(y_test, y_pred),6)
    adjusted_r_squared = round(1 - (1-r_squared)*(len(y)-1)/(len(y)-X.shape[1]-1),6)
    print('Adj R Square: ', adjusted_r_squared)
    print('------------------------------------------------------------------------------------------------------------')
    Results.loc[len(Results)] = [
    type(models).__name__,
    metrics.mean_absolute_error(y_test, y_pred),
    metrics.mean_squared_error(y_test, y_pred),
    np.sqrt(metrics.mean_squared_error(y_test, y_pred)),
    metrics.r2_score(y_test, y_pred),
    np.log(np.sqrt(metrics.mean_squared_error(y_test, y_pred))),
    MAPE(y_test, y_pred),
    adjusted_r_squared
    ]

Model Name:  LinearRegression()
Mean Absolute Error (MAE): 0.269
Mean Squared Error (MSE): 0.119
Root Mean Squared Error (RMSE): 0.345
R2_score: 0.512035
Root Mean Squared Log Error (RMSLE): -1.066
Mean Absolute Percentage Error (MAPE): inf %
Adj R Square:  0.511808
------------------------------------------------------------------------------------------------------------


/tmp/ipykernel_11101/2838521927.py:3: RuntimeWarning: divide by zero encountered in divide
  return np.mean(np.abs((y_test - y_pred) / y_test)) * 100
/tmp/ipykernel_11101/2838521927.py:3: RuntimeWarning: divide by zero encountered in divide
  return np.mean(np.abs((y_test - y_pred) / y_test)) * 100


Model Name:  DecisionTreeRegressor()
Mean Absolute Error (MAE): 0.224
Mean Squared Error (MSE): 0.224
Root Mean Squared Error (RMSE): 0.474
R2_score: 0.077093
Root Mean Squared Log Error (RMSLE): -0.747
Mean Absolute Percentage Error (MAPE): nan %
Adj R Square:  0.076663
------------------------------------------------------------------------------------------------------------


/tmp/ipykernel_11101/2838521927.py:3: RuntimeWarning: divide by zero encountered in divide
  return np.mean(np.abs((y_test - y_pred) / y_test)) * 100
/tmp/ipykernel_11101/2838521927.py:3: RuntimeWarning: invalid value encountered in divide
  return np.mean(np.abs((y_test - y_pred) / y_test)) * 100
/tmp/ipykernel_11101/2838521927.py:3: RuntimeWarning: divide by zero encountered in divide
  return np.mean(np.abs((y_test - y_pred) / y_test)) * 100
/tmp/ipykernel_11101/2838521927.py:3: RuntimeWarning: invalid value encountered in divide
  return np.mean(np.abs((y_test - y_pred) / y_test)) * 100


Model Name:  RandomForestRegressor()
Mean Absolute Error (MAE): 0.227
Mean Squared Error (MSE): 0.119
Root Mean Squared Error (RMSE): 0.345
R2_score: 0.509263
Root Mean Squared Log Error (RMSLE): -1.063
Mean Absolute Percentage Error (MAPE): nan %
Adj R Square:  0.509034
------------------------------------------------------------------------------------------------------------


...

/tmp/ipykernel_11101/2838521927.py:3: RuntimeWarning: divide by zero encountered in divide
  return np.mean(np.abs((y_test - y_pred) / y_test)) * 100
/tmp/ipykernel_11101/2838521927.py:3: RuntimeWarning: invalid value encountered in divide
  return np.mean(np.abs((y_test - y_pred) / y_test)) * 100
/tmp/ipykernel_11101/2838521927.py:3: RuntimeWarning: divide by zero encountered in divide
  return np.mean(np.abs((y_test - y_pred) / y_test)) * 100
/tmp/ipykernel_11101/2838521927.py:3: RuntimeWarning: invalid value encountered in divide
  return np.mean(np.abs((y_test - y_pred) / y_test)) * 100
/home/mobina-tkd-85/Ml Project/Student-Depression/venv/lib/python3.12/site-packages/sklearn/ensemble/_gb.py:666: FutureWarning: The parameter `criterion` is deprecated and will be removed in 1.11. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


Model Name:  GradientBoostingRegressor(criterion='friedman_mse')
Mean Absolute Error (MAE): 0.245
Mean Squared Error (MSE): 0.114
Root Mean Squared Error (RMSE): 0.338
R2_score: 0.531292
Root Mean Squared Log Error (RMSLE): -1.086
Mean Absolute Percentage Error (MAPE): inf %
Adj R Square:  0.531074
------------------------------------------------------------------------------------------------------------


/tmp/ipykernel_11101/2838521927.py:3: RuntimeWarning: divide by zero encountered in divide
  return np.mean(np.abs((y_test - y_pred) / y_test)) * 100
/tmp/ipykernel_11101/2838521927.py:3: RuntimeWarning: divide by zero encountered in divide
  return np.mean(np.abs((y_test - y_pred) / y_test)) * 100


Model Name:  BaggingRegressor()
Mean Absolute Error (MAE): 0.227
Mean Squared Error (MSE): 0.13
Root Mean Squared Error (RMSE): 0.361
R2_score: 0.465161
Root Mean Squared Log Error (RMSLE): -1.02
Mean Absolute Percentage Error (MAPE): nan %
Adj R Square:  0.464912
------------------------------------------------------------------------------------------------------------
Model Name:  Ridge()
Mean Absolute Error (MAE): 0.269
Mean Squared Error (MSE): 0.119
Root Mean Squared Error (RMSE): 0.344
R2_score: 0.5121
Root Mean Squared Log Error (RMSLE): -1.066
Mean Absolute Percentage Error (MAPE): inf %
Adj R Square:  0.511873
------------------------------------------------------------------------------------------------------------
Model Name:  Lasso(alpha=0.1)
Mean Absolute Error (MAE): 0.398
Mean Squared Error (MSE): 0.177
Root Mean Squared Error (RMSE): 0.42
R2_score: 0.273293
Root Mean Squared Log Error (RMSLE): -0.867
Mean Absolute Percentage Error (MAPE): inf %
Adj R Square:  0.272954

/tmp/ipykernel_11101/2838521927.py:3: RuntimeWarning: divide by zero encountered in divide
  return np.mean(np.abs((y_test - y_pred) / y_test)) * 100
/tmp/ipykernel_11101/2838521927.py:3: RuntimeWarning: invalid value encountered in divide
  return np.mean(np.abs((y_test - y_pred) / y_test)) * 100
/tmp/ipykernel_11101/2838521927.py:3: RuntimeWarning: divide by zero encountered in divide
  return np.mean(np.abs((y_test - y_pred) / y_test)) * 100
/tmp/ipykernel_11101/2838521927.py:3: RuntimeWarning: invalid value encountered in divide
  return np.mean(np.abs((y_test - y_pred) / y_test)) * 100
/tmp/ipykernel_11101/2838521927.py:3: RuntimeWarning: divide by zero encountered in divide
  return np.mean(np.abs((y_test - y_pred) / y_test)) * 100
/tmp/ipykernel_11101/2838521927.py:3: RuntimeWarning: divide by zero encountered in divide
  return np.mean(np.abs((y_test - y_pred) / y_test)) * 100
/tmp/ipykernel_11101/2838521927.py:3: RuntimeWarning: divide by zero encountered in divide
  return np

In [25]:
Results

,Model,Mean_Absolute_Error_MAE,Mean_Squared_Error_MSE,Root_Mean_Squared_Error_RMSE,R2_Score,Root_Mean_Squared_Log_Error_RMSLE,Mean_Absolute_Percentage_Error_MAPE,Adjusted_R2_Score
0,LinearRegression,0.268624,0.118681,0.344501,0.512035,-1.065660,inf,0.511808
1,DecisionTreeRegressor,0.224465,0.224465,0.473778,0.077093,-0.747017,NaN,0.076663
2,RandomForestRegressor,0.226953,0.119355,0.345478,0.509263,-1.062827,NaN,0.509034
3,KNeighborsRegressor,0.234381,0.136270,0.369148,0.439713,-0.996557,NaN,0.439452
4,ExtraTreesRegressor,0.223802,0.125975,0.354930,0.482044,-1.035836,NaN,0.481803
5,GradientBoostingRegressor,0.245474,0.113997,0.337634,0.531292,-1.085792,inf,0.531074
6,BaggingRegressor,0.227273,0.130081,0.360668,0.465161,-1.019798,NaN,0.464912
7,Ridge,0.268623,0.118665,0.344478,0.512100,-1.065726,inf,0.511873
8,Lasso,0.397790,0.176746,0.420412,0.273293,-0.866520,inf,0.272954


In [26]:
Results.describe()

/home/mobina-tkd-85/Ml Project/Student-Depression/venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4608: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
/home/mobina-tkd-85/Ml Project/Student-Depression/venv/lib/python3.12/site-packages/pandas/core/nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,Mean_Absolute_Error_MAE,Mean_Squared_Error_MSE,Root_Mean_Squared_Error_RMSE,R2_Score,Root_Mean_Squared_Log_Error_RMSLE,Mean_Absolute_Percentage_Error_MAPE,Adjusted_R2_Score
count,9.000000,9.000000,9.000000,9.000000,9.000000,4.0,9.000000
mean,0.257487,0.140471,0.372336,0.422444,-0.993970,inf,0.422175
std,0.055545,0.036749,0.045453,0.151096,0.113532,NaN,0.151166
min,0.223802,0.113997,0.337634,0.077093,-1.085792,inf,0.076663
25%,0.226953,0.118681,0.344501,0.439713,-1.065660,NaN,0.439452
50%,0.234381,0.125975,0.354930,0.482044,-1.035836,NaN,0.481803
75%,0.268623,0.136270,0.369148,0.512035,-0.996557,NaN,0.511808
max,0.397790,0.224465,0.473778,0.531292,-0.747017,inf,0.531074


From the Above Results, The Top  Model by comparing Errors , is DecisionTreeRegresso

### Training the model with decision tree regressor algorythm

In [27]:
modeldcr.fit(X_train, y_train)
y_pred = modeldcr.predict(X_test)
output = pd.DataFrame({'Actual Depression Situation' : y_test, 'Predicted Depression Situation': y_pred})

output['Right Output'] = (
    output['Actual Depression Situation'] ==
    output['Predicted Depression Situation']
).astype(int)
result=X_test.merge(output,left_index=True,right_index=True)

In [28]:
result

,num_pipeline__Age,num_pipeline__Academic Pressure,num_pipeline__CGPA,num_pipeline__Study Satisfaction,num_pipeline__Work/Study Hours,num_pipeline__Financial Stress,ordinal_pipeline__Sleep Duration,ordinal_pipeline__Dietary Habits,cat_pipeline__Gender_Female,cat_pipeline__Gender_Male,...,cat_pipeline__Degree_MSc,cat_pipeline__Degree_Others,cat_pipeline__Degree_PhD,cat_pipeline__Have you ever had suicidal thoughts ?_No,cat_pipeline__Have you ever had suicidal thoughts ?_Yes,cat_pipeline__Family History of Mental Illness_No,cat_pipeline__Family History of Mental Illness_Yes,Actual Depression Situation,Predicted Depression Situation,Right Output
2,0.446187,-0.818004,1.344150,1.509960,-1.654988,1.289711,0.0,2.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0,0.0,1
4,0.649236,-0.095812,1.575391,0.038280,0.499932,-1.497295,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0,0.0,1
5,0.649236,0.626381,-0.498983,-1.433399,-0.577528,-0.103792,2.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0,0.0,1
8,0.243139,-0.095812,-0.648610,0.038280,0.769297,1.289711,1.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1,1.0,1
9,0.649236,-1.540196,-1.736806,0.774120,-1.385623,0.592960,3.0,1.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1,1.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8354,-0.366007,1.348573,0.378375,-0.697559,1.038662,-0.800544,2.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1,1.0,1
8358,0.446187,-0.095812,0.330767,-0.697559,-0.577528,-0.800544,3.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1,1.0,1
8359,-0.975153,1.348573,-1.369540,0.038280,0.499932,1.289711,2.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1,1.0,1
8362,-0.366007,-0.095812,0.738840,-0.697559,-1.924353,0.592960,2.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1,0.0,0


In [29]:
result['Right Output'].value_counts()

Right Output
1    1967
0     610
Name: count, dtype: int64

## Making pkl files

In [34]:
import pickle as pkl

pkl.dump(modeldcr,open('model.pkl','wb'))

pkl.dump(preprocessor,open('preprocessor.pkl','wb'))